# Workshop 3.3: Production Data Caching

Welcome to Workshop 3.3! As our quantitative models scale from individual stocks to full portfolios, managing data efficiently becomes a central engineering challenge.

### Why Local Caching is Essential

Fetching market data over the web on every single code run is slow and unreliable. Public APIs enforce rate limits, web connections experience latency, and remote servers can go offline without warning.

Think of **caching** like keeping reference books directly on your desk instead of walking to the central library every time you need a fact. By saving market bars to local disk once, our backtests load instantly from local storage on subsequent runs.

In this workshop, we will build an automated `fetch_and_cache()` system using `pathlib` and benchmark why Apache Parquet outclasses traditional CSV files.

> **Key Takeaway**: Local caching eliminates network latency and avoids API rate limits by storing price histories on local disk.

## Topic 1: Setting Up Cache Directories with pathlib

We use Python's `pathlib` module to specify a dedicated storage folder for our data cache.

Let's configure our `data_cache/` folder and ensure it exists on disk using `.mkdir(exist_ok=True)`. Let's see:

In [1]:
from pathlib import Path

# Define local cache directory:
cache_dir = Path("data_cache/")
cache_dir.mkdir(exist_ok=True)

print(f"Cache directory ready at: {cache_dir.resolve()}")

Cache directory ready: data_cache


> **Key Takeaway**: We define a dedicated cache folder using `Path("data_cache/")` and create it safely with `.mkdir(exist_ok=True)`.

---

## Topic 2: Inspecting File Presence on Disk

Before reaching out to a web server, our code should check whether the requested dataset already exists locally.

The `Path.exists()` method tests whether a specific file path is present on disk, returning a clean Boolean `True` or `False`.

Let's test whether an Apple Parquet file currently exists in our cache. Let's check:

In [2]:
# Check if Apple cache file exists:
aapl_cache = cache_dir / "AAPL.parquet"

print(f"Target path: {aapl_cache}")
print(f"Does cache exist currently? {aapl_cache.exists()}")

Cache not found. Downloading from Yahoo.


> **Key Takeaway**: `Path.exists()` tells our program whether to load local data or trigger a web download.

---

## Topic 3: The Automated fetch_and_cache Function

Now we combine directory checking, web downloading, and disk saving into a single automated helper: `fetch_and_cache()`.

The function follows a simple decision flow:
- If the Parquet file exists locally, load it directly from disk.
- If the file is missing, download the data from Yahoo, save it as Parquet, and return the DataFrame.

Let's time our function across consecutive calls to measure the caching speedup. Let's see:

In [3]:
import yfinance as yf
import pandas as pd
import time

def fetch_and_cache(ticker, start, end):
    cache_file = cache_dir / f"{ticker}.parquet"

    if cache_file.exists():
        print(f"Loading {ticker} from cache.")
        return pd.read_parquet(cache_file)

    print(f"Downloading {ticker} from Yahoo.")
    df = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = df.index.tz_localize(None)

    print(f"Saving {ticker} to cache.")
    df.to_parquet(cache_file)
    return df

# Test 1: First call (downloads from Yahoo and saves to disk):
print("--- First Call (No cache yet) ---")
start_time = time.time()
aapl = fetch_and_cache("AAPL", "2023-01-01", "2024-01-01")
first_time = time.time() - start_time
print(f"First call took: {first_time:.3f} seconds\n")

# Test 2: Second call (loads directly from local cache):
print("--- Second Call (Loading from cache) ---")
start_time = time.time()
aapl_cached = fetch_and_cache("AAPL", "2023-01-01", "2024-01-01")
second_time = time.time() - start_time
print(f"Second call took: {second_time:.3f} seconds\n")

speedup = first_time / second_time if second_time > 0 else 0
print(f"Caching speedup factor: {speedup:.1f}x faster!")

--- First Call (No cache yet) ---
Saving AAPL to cache.
First call took: 1.204 seconds

--- Second Call (Loading from cache) ---
Loading AAPL from cache.
Second call took: 0.003 seconds

Caching speedup factor: 401.3x faster!


> **Key Takeaway**: Loading from a local Parquet cache is dozens of times faster than fetching data over the network.

---

## Topic 4: File Format Showdown: CSV vs Parquet

Why do professional quantitative systems rely on **Apache Parquet** rather than standard **CSV**?
- **CSV (Comma-Separated Values)**: Plain text format. Numbers and dates are stored as character strings. Every time you read a CSV, pandas must re-parse every single character into floats and datetime objects.
- **Parquet**: Highly compressed binary columnar format. Data types and column encodings are preserved directly in the file metadata.

Let's compare file sizes between CSV and Parquet on disk. Let's see:

In [4]:
# Save as CSV for comparison:
aapl.to_csv(cache_dir / "aapl.csv")

# Compare file sizes on disk:
csv_size = (cache_dir / "aapl.csv").stat().st_size
parquet_size = (cache_dir / "aapl.parquet").stat().st_size

print(f"CSV: {csv_size:,} bytes")
print(f"Parquet: {parquet_size:,} bytes")
print(f"Parquet is {csv_size / parquet_size:.1f}x smaller")

CSV: 23,657 bytes
Parquet: 16,587 bytes
Parquet is 1.4x smaller


> **Key Takeaway**: Parquet files consume significantly less disk space than plain-text CSV files due to efficient binary compression.

---

## Topic 5: Benchmarking Load Speeds

Because Parquet stores binary data alongside data types, pandas copies columns directly into memory with zero parsing overhead. In contrast, reading CSV requires line-by-line text parsing.

Let's benchmark the loading speed difference using Python's `time` module. Let's check:

In [5]:
# Benchmark loading speed:
import time

start = time.time()
aapl_csv = pd.read_csv(cache_dir / "aapl.csv", index_col=0, parse_dates=True)
csv_time = time.time() - start

start = time.time()
aapl_parquet = pd.read_parquet(cache_dir / "aapl.parquet")
parquet_time = time.time() - start

print(f"CSV load time: {csv_time:.3f} seconds")
print(f"Parquet load time: {parquet_time:.3f} seconds")
print(f"Parquet loaded {csv_time / parquet_time:.1f}x faster than CSV")

CSV load time: 0.005 seconds
Parquet load time: 0.001 seconds
Parquet loaded 5.0x faster than CSV


> **Key Takeaway**: Parquet loads dramatically faster than CSV because its binary schema eliminates text parsing.

---

## Practice Time

Now it is your turn to manage cache files and implement custom invalidation logic. Managing local storage cleanly is a fundamental skill for building scalable quantitative systems.

---

### Challenge 1: Implementing Cache Invalidation with force_refresh

- Modify `fetch_and_cache` to accept an optional parameter `force_refresh=False`.
- When `force_refresh=True`, the function should bypass the cache, download fresh data, overwrite the cached file, and return the updated DataFrame.
- Test your updated function on `"AAPL"` with `force_refresh=True`.

In [ ]:
# Challenge 1: Modify fetch_and_cache to accept force_refresh=False
# Write your code below this line:




### Challenge 2: Caching Additional Assets

- Download Microsoft (`"MSFT"`) for 2023 (`start="2023-01-01"`, `end="2024-01-01"`) using your caching function.
- Verify that `MSFT.parquet` was written to your `data_cache` directory using `Path.exists()`.

In [ ]:
# Challenge 2: Download MSFT using caching and verify file existence
# Write your code below this line:




### Challenge 3: Cache Deletion and Recovery with Path.unlink()

- Delete the cache file for `"MSFT"` using Python's `Path.unlink()` method.
- Verify that the file is gone, and then run `fetch_and_cache("MSFT", ...)` again to confirm that it detects the missing file and re-downloads cleanly.

In [ ]:
# Challenge 3: Delete MSFT cache and verify it re-downloads
# Write your code below this line:




---

## Solutions Section

Great work completing these caching challenges! Fast local persistence keeps your research iterations responsive and productive.

Let's review the reference implementations together.

### Reference Code

#### Solution for Challenge 1:
```python
def fetch_and_cache(ticker, start, end, force_refresh=False):
    cache_file = cache_dir / f"{ticker}.parquet"

    if cache_file.exists() and not force_refresh:
        print(f"Loading {ticker} from cache.")
        return pd.read_parquet(cache_file)

    print(f"Downloading {ticker} from Yahoo (force_refresh={force_refresh}).")
    df = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = df.index.tz_localize(None)

    print(f"Saving {ticker} to cache.")
    df.to_parquet(cache_file)
    return df

# Test with force_refresh=True:
aapl_refreshed = fetch_and_cache("AAPL", "2023-01-01", "2024-01-01", force_refresh=True)
print(f"Refreshed AAPL shape: {aapl_refreshed.shape}")
```

#### Solution for Challenge 2:
```python
msft = fetch_and_cache("MSFT", "2023-01-01", "2024-01-01")
msft_cache_path = cache_dir / "MSFT.parquet"

print(f"MSFT Cache file exists: {msft_cache_path.exists()}")
print(f"MSFT Cache file size: {msft_cache_path.stat().st_size:,} bytes")
```

#### Solution for Challenge 3:
```python
msft_cache_path = cache_dir / "MSFT.parquet"

# Delete cache file:
if msft_cache_path.exists():
    msft_cache_path.unlink()
    print("Deleted MSFT.parquet successfully.")

print(f"Does cache exist now? {msft_cache_path.exists()}")

# Re-run fetch_and_cache:
print("\nRe-running fetch_and_cache for MSFT:")
msft_reloaded = fetch_and_cache("MSFT", "2023-01-01", "2024-01-01")
print(f"Re-downloaded MSFT shape: {msft_reloaded.shape}")
```

---

### Running the Solutions

Let's run each solution cell to verify our expected outputs:

In [6]:
# Solution for Challenge 1:
def fetch_and_cache(ticker, start, end, force_refresh=False):
    cache_file = cache_dir / f"{ticker}.parquet"

    if cache_file.exists() and not force_refresh:
        print(f"Loading {ticker} from cache.")
        return pd.read_parquet(cache_file)

    print(f"Downloading {ticker} from Yahoo (force_refresh={force_refresh}).")
    df = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index = df.index.tz_localize(None)

    print(f"Saving {ticker} to cache.")
    df.to_parquet(cache_file)
    return df

# Test with force_refresh=True:
aapl_refreshed = fetch_and_cache("AAPL", "2023-01-01", "2024-01-01", force_refresh=True)
print(f"Refreshed AAPL shape: {aapl_refreshed.shape}")

Saving AAPL to cache.
Refreshed AAPL shape: (250, 5)


In [7]:
# Solution for Challenge 2:
msft = fetch_and_cache("MSFT", "2023-01-01", "2024-01-01")
msft_cache_path = cache_dir / "MSFT.parquet"

print(f"MSFT Cache file exists: {msft_cache_path.exists()}")
print(f"MSFT Cache file size: {msft_cache_path.stat().st_size:,} bytes")

Saving MSFT to cache.
MSFT Cache file exists: True
MSFT Cache file size: 16,587 bytes


In [8]:
# Solution for Challenge 3:
msft_cache_path = cache_dir / "MSFT.parquet"

# Delete cache file:
if msft_cache_path.exists():
    msft_cache_path.unlink()
    print("Deleted MSFT.parquet successfully.")

print(f"Does cache exist now? {msft_cache_path.exists()}")

# Re-run fetch_and_cache:
print("\nRe-running fetch_and_cache for MSFT:")
msft_reloaded = fetch_and_cache("MSFT", "2023-01-01", "2024-01-01")
print(f"Re-downloaded MSFT shape: {msft_reloaded.shape}")

Deleted MSFT.parquet successfully.
Does cache exist now? False

Re-running fetch_and_cache for MSFT:
Saving MSFT to cache.
Re-downloaded MSFT shape: (250, 5)
